# Prediction time and table definitions

## 1. Start Spark and load the files

In [1]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "../data"
attributes = spark.read.csv(f"{DATA_DIR}/features_attributes.csv", header=True, inferSchema=True)
financials = spark.read.csv(f"{DATA_DIR}/features_financials.csv", header=True, inferSchema=True)
clickstream = spark.read.csv(f"{DATA_DIR}/feature_clickstream.csv", header=True, inferSchema=True)
lms = spark.read.csv(f"{DATA_DIR}/lms_loan_daily.csv", header=True, inferSchema=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 11:27:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Lab 2 silver logic: mob and dpd
def add_dpd(loans):
    loans = loans.withColumn("mob", col("installment_num"))
    loans = loans.withColumn("installments_missed", F.ceil(col("overdue_amt") / col("due_amt")).cast("int")).fillna(0, subset=["installments_missed"])
    loans = loans.withColumn("first_missed_date", F.when(col("installments_missed") > 0, F.add_months(col("snapshot_date"), -1 * col("installments_missed"))))
    loans = loans.withColumn("dpd", F.when(col("overdue_amt") > 0.0, F.datediff(col("snapshot_date"), col("first_missed_date"))).otherwise(0))
    return loans

loans = add_dpd(lms)

## 2. One applicant's timeline

In [3]:
# Find a 2023-06 applicant who defaults at mob 6
candidates = loans.filter(col("mob") == 6).withColumn("label", (col("dpd") >= 30).cast("int")) \
    .filter((col("label") == 1) & (F.date_format(col("loan_start_date"), "yyyy-MM") == "2023-06")) \
    .select("Customer_ID")

candidates.show(10, truncate=False)

+-----------+
|Customer_ID|
+-----------+
|CUS_0x10ee |
|CUS_0x1136 |
|CUS_0x142f |
|CUS_0x145c |
|CUS_0x14ca |
|CUS_0x166c |
|CUS_0x1766 |
|CUS_0x1972 |
|CUS_0x1c99 |
|CUS_0x1db0 |
+-----------+
only showing top 10 rows



In [4]:
customer = "CUS_0x10ee"
attributes.filter(col("Customer_ID") == customer).show()
financials.filter(col("Customer_ID") == customer).select("Customer_ID", "Annual_Income", "Outstanding_Debt", "Credit_Mix", "snapshot_date").show()

+-----------+-------+---+-----------+----------+-------------+
|Customer_ID|   Name|Age|        SSN|Occupation|snapshot_date|
+-----------+-------+---+-----------+----------+-------------+
| CUS_0x10ee|Hudsonb|30_|699-80-2426|Journalist|   2023-06-01|
+-----------+-------+---+-----------+----------+-------------+

+-----------+-------------+----------------+----------+-------------+
|Customer_ID|Annual_Income|Outstanding_Debt|Credit_Mix|snapshot_date|
+-----------+-------------+----------------+----------+-------------+
| CUS_0x10ee|     12660.16|          456.02|  Standard|   2023-06-01|
+-----------+-------------+----------------+----------+-------------+



In [5]:
loans.filter(col("Customer_ID") == customer) \
    .select("snapshot_date", "mob", "due_amt", "paid_amt", "overdue_amt", "balance", "installments_missed", "dpd").orderBy("snapshot_date").show()

+-------------+---+-------+--------+-----------+-------+-------------------+---+
|snapshot_date|mob|due_amt|paid_amt|overdue_amt|balance|installments_missed|dpd|
+-------------+---+-------+--------+-----------+-------+-------------------+---+
|   2023-06-01|  0|    0.0|     0.0|        0.0|10000.0|                  0|  0|
|   2023-07-01|  1| 1000.0|  1000.0|        0.0| 9000.0|                  0|  0|
|   2023-08-01|  2| 1000.0|     0.0|     1000.0| 9000.0|                  1| 31|
|   2023-09-01|  3| 1000.0|  2000.0|        0.0| 7000.0|                  0|  0|
|   2023-10-01|  4| 1000.0|     0.0|     1000.0| 7000.0|                  1| 30|
|   2023-11-01|  5| 1000.0|     0.0|     2000.0| 7000.0|                  2| 61|
|   2023-12-01|  6| 1000.0|     0.0|     3000.0| 7000.0|                  3| 91|
|   2024-01-01|  7| 1000.0|     0.0|     4000.0| 7000.0|                  4|122|
|   2024-02-01|  8| 1000.0|     0.0|     5000.0| 7000.0|                  5|153|
|   2024-03-01|  9| 1000.0| 

In [6]:
application_date = attributes.filter(col("Customer_ID") == customer).first()["snapshot_date"]
print("application month:", application_date)

clickstream.filter(col("Customer_ID") == customer) \
    .withColumn("when", F.when(col("snapshot_date") < F.lit(application_date), "1: before")
                         .when(col("snapshot_date") == F.lit(application_date), "2: same month")
                         .otherwise("3: after")) \
    .groupBy("when").agg(F.count("*").alias("months"), F.min("snapshot_date").alias("first"), F.max("snapshot_date").alias("last")).orderBy("when").show()

application month: 2023-06-01
+-------------+------+----------+----------+
|         when|months|     first|      last|
+-------------+------+----------+----------+
|    1: before|     5|2023-01-01|2023-05-01|
|2: same month|     1|2023-06-01|2023-06-01|
|     3: after|    18|2023-07-01|2024-12-01|
+-------------+------+----------+----------+



+-------------+------+----------+----------+
|         when|months|     first|      last|
+-------------+------+----------+----------+
|    1. before|     5|2023-01-01|2023-05-01|
|2. same month|     1|2023-06-01|2023-06-01|
|     3. after|    18|2023-07-01|2024-12-01|
+-------------+------+----------+----------+



**Findings:**
- CUS_0x10ee applied in 2023-06: the attributes and financials snapshot is dated 2023-06-01, the loan start month
- The loan missed installment 2, caught up at 3, then missed every installment from 4
- At mob 6 (2023-12) it was 3 installments (91 days) overdue, so Lab 2's rule labels it bad, and the label is known in 2023-12
- Of this customer's 24 clickstream data months, 5 are before the application month, 1 is the same month and 18 are after it
- At application, only the 2023-06 attributes and financials and the clickstream up to 2023-06 exist

## 3. Joining features at the label date

In [7]:
labels = loans.filter(col("mob") == 6).withColumn("label", (col("dpd") >= 30).cast("int")) \
    .select("loan_id", "Customer_ID", "label", "loan_start_date", "snapshot_date")
print("labels:", labels.count())

for name, df in [("attributes", attributes), ("clickstream", clickstream)]:
    at_label_date = labels.join(df.select("Customer_ID", "snapshot_date"), ["Customer_ID", "snapshot_date"]).count()
    at_application = labels.join(df.select("Customer_ID", col("snapshot_date").alias("loan_start_date")), ["Customer_ID", "loan_start_date"]).count()
    print(f"{name}: {at_label_date} rows match on label date, {at_application} on loan start")

labels: 12500
attributes: 0 rows match on label date, 12500 on loan start
clickstream: 8974 rows match on label date, 8974 on loan start


attributes: 0 rows match on the label date, 12500 on the loan start


clickstream: 8974 rows match on the label date, 8974 on the loan start


**Findings:**
- Joining features on the label's `snapshot_date` (the date Lab 3 joins clickstream on) matches no attributes rows, and matches all 8974 clickstream customers at a month six months after their application
- Joining on loan start matches all 12500 attributes rows and the application-month clickstream row
- Hence, the label store needs `loan_start_date` so that features can be joined at the application date, not the label date

## 4. Lab 2's days past due use calendar months

In [8]:
one_missed = loans.filter(col("installments_missed") == 1)
one_missed.groupBy("dpd").count().orderBy("dpd").show()
one_missed.filter(col("dpd") < 30).groupBy("snapshot_date").count().orderBy("snapshot_date").show()

+---+-----+
|dpd|count|
+---+-----+
| 28|  255|
| 29|  213|
| 30| 1650|
| 31| 2912|
+---+-----+

+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-03-01|   80|
|   2024-03-01|  213|
|   2025-03-01|  175|
+-------------+-----+



+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-03-01|   80|
|   2024-03-01|  213|
|   2025-03-01|  175|
+-------------+-----+



In [9]:
loans.filter(col("mob") == 6).groupBy("installments_missed").count().orderBy("installments_missed").show()

+-------------------+-----+
|installments_missed|count|
+-------------------+-----+
|                  0| 8898|
|                  2|  681|
|                  3| 1133|
|                  4| 1115|
|                  5|  541|
|                  6|  132|
+-------------------+-----+



**Findings:**
- One missed installment gives 28 to 31 days past due, depending on the length of the previous month; 468 rows (all March snapshots) fall below 30.
- At mob 6 a loan has 0 or at least 2 missed installments, never exactly 1, so the 30-dpd label at mob 6 is not affected -> Keep Lab 2's logic.

## 5. Decisions from the EDA: table definitions

**Feature store (gold)**
- One row per loan application, keyed on `Customer_ID` and `snapshot_date` (the application month)
- Built from silver attributes, financials and clickstream
- Time rule: attributes and financials come from the application month; clickstream is averaged over months up to and including the application month

**Label store (gold)**
- One row per loan, keyed on `loan_id` (with `Customer_ID` and `loan_start_date` also kept)
- Built from silver loan_daily
- Label = 30+ days past due at mob 6 (`30dpd_6mob`). `snapshot_date` on this table is the label date, loan start + 6 months, not the application date.

**Bronze and silver (attributes, financials, clickstream, loan_daily)**
- One row per source row, keyed on the source's own keys
- Built from the raw CSVs
- Time rule: one partition per snapshot month

**Assumptions and rules**
- The application-month snapshot (dated the 1st) is known at application. Clickstream dated the application month is treated the same way; later months are never used.
- Applicants without clickstream (loans from 2024-07) keep missing clickstream features: no zero fill and no `has_clickstream` flag
- Features and labels join on `Customer_ID` and `feature_store.snapshot_date = label_store.loan_start_date`
- A model trained at date T may only use labels with `snapshot_date` up to T
- Loan repayment columns feed the label store only